In [1]:
list.of.packages <- c("ggplot2", "Rcpp", "grf", "caret", "mltools", "rpart", "minpack.lm", "doParallel", "rattle", "anytime","rlist")
list.of.packages <- c(list.of.packages, "zoo", "dtw", "foreach", "evaluate","rlist","data.table","plyr","dplyr")


new.packages <- list.of.packages[!(list.of.packages %in% installed.packages()[,"Package"])]
if(length(new.packages)) install.packages(new.packages, lib='/home/zwang937/local/R_libs', repos="http://cran.us.r-project.org", dependencies = TRUE, INSTALL_opts = '--no-lock')


lapply(list.of.packages, require, character.only = TRUE)


Loading required package: ggplot2

Loading required package: Rcpp

Loading required package: grf

Loading required package: caret

Loading required package: lattice

Loading required package: mltools

Loading required package: rpart

Loading required package: minpack.lm

Loading required package: doParallel

Loading required package: foreach

Loading required package: iterators

Loading required package: parallel

Loading required package: rattle

Loading required package: tibble

Loading required package: bitops

Rattle: A free graphical interface for data science with R.
Version 5.5.1 Copyright (c) 2006-2021 Togaware Pty Ltd.
Type 'rattle()' to shake, rattle, and roll your data.

Loading required package: anytime

Loading required package: rlist

Loading required package: zoo


Attaching package: ‘zoo’


The following objects are masked from ‘package:base’:

    as.Date, as.Date.numeric


Loading required package: dtw

Loading required package: proxy


Attaching package: ‘proxy’


Th

[[1]]
[1] TRUE

[[2]]
[1] TRUE

[[3]]
[1] TRUE

[[4]]
[1] TRUE

[[5]]
[1] TRUE

[[6]]
[1] TRUE

[[7]]
[1] TRUE

[[8]]
[1] TRUE

[[9]]
[1] TRUE

[[10]]
[1] TRUE

[[11]]
[1] TRUE

[[12]]
[1] TRUE

[[13]]
[1] TRUE

[[14]]
[1] TRUE

[[15]]
[1] TRUE

[[16]]
[1] TRUE

[[17]]
[1] TRUE

[[18]]
[1] TRUE

[[19]]
[1] TRUE

In [2]:
# Initialize a cluster for parallel processing
cl <- makeCluster(detectCores())

# Register the cluster
registerDoParallel(cl)

In [3]:
TLGRF_RDS_directory <- "../../data/output/grf_windowsize=2_numtrees=200"
file_pattern <- "grf_stateforest_cutoff=%d.rds"
days_from_start_list <-seq(100, 1000, by=100)
# Define a function to load RDS objects
load_rds <- function(file) {
  rds_file <- sprintf(file_pattern, file)
  readRDS(file.path(TLGRF_RDS_directory, rds_file))
}

# Load RDS objects in parallel
results <- foreach(day = days_from_start_list, .packages = "utils") %dopar% {
  print(paste0("Loading day=", as.character(day)))
  grf_object <- load_rds(day)
  grf::variable_importance(grf_object)
}

In [4]:
grf_object <- load_rds(100)  # Assuming you have already loaded the RDS object

# Extract the names of X.orig
names <- colnames(grf_object$X.orig)


In [5]:
names

[1] "fips"                                                                                                                            
  [2] "log_rolled_cases.y"                                                                                                              
  [3] "LAT"                                                                                                                             
  [4] "LON"                                                                                                                             
  [5] "AREA_SQMI"                                                                                                                       
  [6] "E_TOTPOP"                                                                                                                        
  [7] "E_HU"                                                                                                                            
  [8] "E_HH"                                                                                                                            
  [9] "E_POV"                                                                                                                           
 [10] "E_UNEMP"                                                                                                                         
 [11] "E_PCI"                                                                                                                           
 [12] "E_NOHSDP"                                                                                                                        
 [13] "E_AGE65"                                                                                                                         
 [14] "E_AGE17"                                                                                                                         
 [15] "E_DISABL"                                                                                                                        
 [16] "E_SNGPNT"                                                                                                                        
 [17] "E_MINRTY"                                                                                                                        
 [18] "E_LIMENG"                                                                                                                        
 [19] "E_MUNIT"                                                                                                                         
 [20] "E_MOBILE"                                                                                                                        
 [21] "E_CROWD"                                                                                                                         
 [22] "E_NOVEH"                                                                                                                         
 [23] "E_GROUPQ"                                                                                                                        
 [24] "EP_POV"                                                                                                                          
 [25] "MP_POV"                                                                                                                          
 [26] "EP_UNEMP"                                                                                                                        
 [27] "MP_UNEMP"                                                                                                                        
 [28] "EP_PCI"                                                                                                                          
 [29] "MP_PCI"                                                                                                                          
 [30] "EP_NOHSDP"            

In [6]:
length(results)

[1] 10

In [7]:
results[1]

1.100312e-04
3.326609e-04
7.478452e-04
1.604621e-03
5.501559e-05
6.803594e-04
1.668439e-03
1.021823e-03
7.353750e-04
4.686595e-03
5.552907e-04


In [8]:
result_matrix <- matrix(NA, nrow = length(days_from_start_list), ncol = 473)
for (i in 1:length(results)) {
  result_matrix[i, ] <- as.vector(results[[i]])
}

In [9]:
colnames(result_matrix) <- names


In [10]:
fwrite(result_matrix, "feature_importance.csv", row.names=FALSE)

x being coerced from class: matrix to data.table



In [11]:
#write_csv(result_matrix, file = "feature_importance.csv", row.names = FALSE)